In [ ]:
%%sql
select * from edw.tbl_fact_policy_quotes limit 100

In [ ]:
%%sql
select * from edw.tbl_fact_policy_sales limit 100

In [ ]:
%%sql
select * from edw.tbl_fact_policy_renewals limit 100

In [ ]:
%%sql
select * from dlk.hfq_quotedetails limit 100

In [ ]:
%%sql
select * from dlk.genesys_session_summary 
Where conversationStartTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' and direction = 'outbound' limit 100

Create Event Stream

In [ ]:
Create or Replace Table ods.EpisodeEventStream 
(
    EventId     BigInt IDENTITY,
    EventSourceId   Int,
    EventSourceReference    Varchar(128),
    SourceQuoteReference    Varchar(128),
    SourcePolicyReference   Varchar(128),
    SourceCustomerReference Varchar(128),
    SourceSystemId          int,
    PolicyTypeGroup         varchar(50),
    EventDateTime           TIMESTAMP,
    EventDate               Date,
    EventTypeId             VARCHAR(10),
    EventDescription        varchar(1024),
    Grain                   varchar(50)
)

Populate step A1

In [8]:
Create or Replace table stg.A1_MotorAcquisitionQuoteInitiated as 
select  QuoteQueryGuid, 
        max(RetrieveSessionToken) RetrieveSessionToken,
        min(case when StepId = 1 then CreatedDateTime else null end) Step1DateTime,
        max(case when StepId in (2,3,4,5) then StepId else null end) Step2To5,
        min(case when StepId in (2,3,4) then CreatedDateTime else null end) FirstStep2to4DateTime,
        min(case when StepId = 5 then CreatedDateTime else null end) FirstStep5DateTime,
        max(case when StepId = 5 then CreatedDateTime else null end) LastStep5DateTime, 
        max(BrokerId) BrokerId
from    dlk.mfq_quotequery 
Where   CreatedDateTime  between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
Group by QuoteQueryGuid

StatementMeta(, 3314ae33-796b-4660-b656-cce564ee428f, 9, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourceQuoteReference,    
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
    QuoteQueryGuid,
    2,
    'Motor',
    date_trunc('MINUTE', Step1DateTime),
    cast(Step1DateTime as date),
    'A1',
    'Motor Acquisition - Quote Initiated',
    'Quote'
FROM
    stg.A1_MotorAcquisitionQuoteInitiated

In [49]:
SELECT count(distinct QuoteQueryGuid)
from    stg.A1_MotorAcquisitionQuoteInitiated  
where   FirstStep5DateTime is null 


StatementMeta(, 3314ae33-796b-4660-b656-cce564ee428f, 50, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourceQuoteReference,    
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
    QuoteQueryGuid,
    2,
    'Motor',
    date_trunc('MINUTE', FirstStep5DateTime),
    cast(FirstStep5DateTime as date),
    'A2',
    'Motor Acquisition - Customer details complete',
    'Quote'
from    stg.A1_MotorAcquisitionQuoteInitiated   
where   FirstStep5DateTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
    /* 
        what about quotes starting inthe previous month but completing this month ? 
            think we should get them too -- els we will have monthly lumps 
    */ 


In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourceQuoteReference,    
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
    QuoteQueryGuid,
    2,
    'Motor',
    date_trunc('MINUTE', FirstStep2to4DateTime),
    cast(FirstStep2to4DateTime as date),
    'A2.F1',
    'Motor Acquisition - Customer details complete',
    'Quote'
from    stg.A1_MotorAcquisitionQuoteInitiated   
where   FirstStep5DateTime is null 
    /* 
        what about quotes starting inthe current month but completing next month ? 
            think we should get them too -- els we will have monthly lumps 
    */ 


## Populate step A0

In [66]:
Create or Replace Table stg.A0_genesys_derived_data as 
select  conversationId, 
        case 
            when originatingDirection = 'inbound' then replace(ani,'tel:+353', '0') 
            when originatingDirection = 'outbound' then replace(dnis,'tel:+353', '0') 
            else null 
        end as CustomerPhoneNumber,
        originatingDirection,          
        conversationStartTime,
        conversationEndTime,
        sessionCount,
        sessionIndex,
        sessionDuration,
        sessionStartTime,
        sessionEndTime,
        queueName,
        purpose,
        transferToId,
        agentAnswered, 
        alertNoAnswer, 
        abandoned, 
        totalAcdWaitDuration, 
        totalAgentAlertDuration, 
        totalAgentHoldDuration, 
        totalAgentTalkDuration
from    dlk.genesys_session_summary
Where	conversationStartTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
;

StatementMeta(, 3314ae33-796b-4660-b656-cce564ee428f, 67, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [7]:
Create or Replace Table stg.A0_genesys_derived_data_filtered as 
select	a.ConversationId, CustomerPhoneNumber,a.sessionIndex,A.conversationStartTime
from	stg.A0_genesys_derived_data	a,
		(select	ConversationId, max(sessionIndex) sessionIndex
		from	stg.A0_genesys_derived_data
		Where	queueName is not null 
		and		originatingDirection = 'inbound'
		Group by ConversationId
		)	b 
where	a.ConversationId = b.ConversationId
and		a.sessionIndex = b.sessionIndex
and		queueName = 'INBOUND_SALES_MOTOR'

/* 
What about these ? 
and		queueName <> 'INBOUND_NON-STANDARD_MOTOR'
and		queueName <> 'OUTBOUND_SALES_MOTOR'
*/

StatementMeta(, 72d9cc6d-00d7-4a7b-9b3d-3b1300e671da, 8, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [3]:
/* A3.E1 */ 
select	count(distinct CustomerPhoneNumber) 
from	stg.A0_genesys_derived_data	a,
		(select	ConversationId, max(sessionIndex) sessionIndex
		from	stg.A0_genesys_derived_data
		Where	queueName is not null 
		and		originatingDirection = 'inbound'
		Group by ConversationId
		)	b 
where	a.ConversationId = b.ConversationId
and		a.sessionIndex = b.sessionIndex
and		queueName = 'OUTBOUND_SALES_MOTOR'

StatementMeta(, 54d0c208-d432-4678-9510-a16144fa7031, 4, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [12]:
Create or Replace Table stg.A0_mfq_derived_data as 
SELECT distinct
    contactmobile,
    QuoteQueryGuid
from
    dlk.mfq_quotequery
Where   CreatedDateTime  between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 


StatementMeta(, 7894b537-5516-446a-a181-0f26a3b21385, 13, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [10]:
Create or Replace table stg.A0_MFQ_genesys_link as 
select  ConversationId,SessionIndex, min(QuoteQueryGuid) ChillSourceQuoteReference
from    stg.A0_genesys_derived_data_filtered a left join stg.A0_mfq_derived_data b
        on a.CustomerPhoneNumber = b.contactmobile
where   b.contactmobile is null
Group by ConversationId,SessionIndex
;

StatementMeta(, 72d9cc6d-00d7-4a7b-9b3d-3b1300e671da, 11, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [11]:
insert INTO
ods.EpisodeEventStream
(        
    EventSourceID,
    SourceQuoteReference,    
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
    ConversationId,
    ChillSourceQuoteReference,
    2,
    'Motor',
    date_trunc('MINUTE', conversationStartTime),
    cast(conversationStartTime as date),
    'A0',
    'Motor Acquisition - straight into call centre',
    'Quote'
FROM
    stg.A0_MFQ_genesys_link

StatementMeta(, 72d9cc6d-00d7-4a7b-9b3d-3b1300e671da, 12, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [73]:
Create or Replace Table stg.A0F1_genesys_inbound_call_duration_summary as 
select	a.ConversationId, min(conversationStartTime) as conversationStartTime, 
            sum(agentAnswered) as agentAnswered, sum(alertNoAnswer) alertNoAnswer, sum(abandoned) abandoned, 
            sum(totalAcdWaitDuration) CallWaitTime, sum(totalAgentAlertDuration) CallRingTime, sum(totalAgentHoldDuration) CallHoldTime, sum(totalAgentTalkDuration) CallSpokenTime
from	stg.A0_genesys_derived_data	a
where	a.queueName = 'INBOUND_SALES_MOTOR'  
Group by ConversationId

StatementMeta(, 3314ae33-796b-4660-b656-cce564ee428f, 74, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [72]:
select count(*) from stg.A0F1_genesys_inbound_call_duration_summary where abandoned = 1 and CallWaitTime > 0 limit 10

StatementMeta(, 3314ae33-796b-4660-b656-cce564ee428f, 73, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    EventSourceID,
    SourceQuoteReference,    
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
    ConversationId,
    null,
    2,
    'Motor',
    date_trunc('MINUTE', conversationStartTime),
    cast(conversationStartTime as date),
    'A0.F1',
    'Motor Acquisition - Call wait time, or fails to make contact',
    'GenesysCall'
FROM
    stg.A0F1_genesys_inbound_call_duration_summary
where   abandoned = 1 
and     CallWaitTime > 0

Populate step A3

In [1]:
Create or Replace table stg.AX_MFQ_genesys_link as 
select  ConversationId,SessionIndex, min(QuoteQueryGuid) ChillSourceQuoteReference
from    stg.A0_genesys_derived_data_filtered a,stg.A0_mfq_derived_data b
Where   replace(a.CustomerPhoneNumber,'tel:+353','0') = replace(b.contactmobile,'tel:+353','0')
Group by ConversationId,SessionIndex
;

StatementMeta(, fcc223e2-df3a-4e4a-8f81-553e7682260a, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
%%sql
select count(*) from stg.AX_MFQ_genesys_link

StatementMeta(, fcc223e2-df3a-4e4a-8f81-553e7682260a, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [3]:
Create or Replace Table stg.MFQInsurersQuoted as 
 WITH base AS (
      SELECT
          qq.QuoteQueryId,
          qq.QuoteQueryGuid,
          qq.CreatedDateTime,
          qq.CoverCode,
          qq.NCDClaimedBonusProtectionType,
          qq.ServiceCallId,
          qr.QuoteResponseId,
          qs.QuoteSummaryId,
          qs.SchemeName,
          qs.Premium,
          qs.Insurer,
          ROW_NUMBER() OVER (
              PARTITION BY qq.QuoteQueryGuid,
                           qq.NCDClaimedBonusProtectionType,
                           qq.CoverCode,
                           COALESCE(qq.ServiceCallId, '0')
              ORDER BY qq.CreatedDateTime DESC, qq.QuoteQueryId DESC
          ) AS latest_service_call_rn
      FROM dlk.MFQ_QuoteQuery qq
      JOIN dlk.MFQ_QuoteResponses qr
        ON qr.RiskID = qq.QuoteQueryId
      JOIN dlk.MFQ_QuoteSummary qs
        ON qs.QuoteResponseId = qr.QuoteResponseId
      JOIN stg.A1_MotorAcquisitionQuoteInitiated z
        ON qq.QuoteQueryGuid = z.QuoteQueryGuid /* focus on this month's guids */ 
      WHERE qr.TransactionID <> '00000000-0000-0000-0000-000000000000'
  ),
  latest_call AS (
      SELECT *
      FROM base
      WHERE latest_service_call_rn = 1
  ),
  lowest_per_insurer AS (
      SELECT *,
          ROW_NUMBER() OVER (
              PARTITION BY QuoteQueryGuid,
                           NCDClaimedBonusProtectionType,
                           CoverCode,
                           COALESCE(ServiceCallId, '0'),
                           Insurer
              ORDER BY Premium
          ) AS insurer_min_premium_rn
      FROM latest_call
      WHERE Premium IS NOT NULL
        AND Premium > 0
        AND SchemeName IS NOT NULL
  ),
  ranked AS (
      SELECT *,
          ROW_NUMBER() OVER (
              PARTITION BY QuoteQueryGuid,
                           NCDClaimedBonusProtectionType,
                           CoverCode,
                           COALESCE(ServiceCallId, '0')
              ORDER BY Premium
          ) AS insurer_rank
      FROM lowest_per_insurer
      WHERE insurer_min_premium_rn = 1
  ),
  rank1 AS (
      SELECT
          QuoteQueryGuid,
          MAX(CASE WHEN insurer_rank = 1 AND CoverCode = '01' AND NCDClaimedBonusProtectionType = 'S' THEN Premium END)
  AS Rank1PremComp,
          MAX(CASE WHEN insurer_rank = 1 AND CoverCode = '02' AND NCDClaimedBonusProtectionType = 'S' THEN Premium END)
  AS Rank1PremTPFT,
          MAX(CASE WHEN insurer_rank = 1 AND CoverCode = '01' AND NCDClaimedBonusProtectionType = 'F' THEN Premium END)
  AS Rank1PremCompNCB,
          MAX(CASE WHEN insurer_rank = 1 AND CoverCode = '02' AND NCDClaimedBonusProtectionType = 'F' THEN Premium END)
  AS Rank1PremTPFTNCB
      FROM ranked
      GROUP BY QuoteQueryGuid
  )
  SELECT
      QuoteQueryGuid,
      CASE
          WHEN COALESCE(Rank1PremComp, 0) <> 0
            OR COALESCE(Rank1PremTPFT, 0) <> 0
            OR COALESCE(Rank1PremCompNCB, 0) <> 0
            OR COALESCE(Rank1PremTPFTNCB, 0) <> 0
          THEN 'Y'
          ELSE 'N'
      END AS InsurersQuoted
  FROM rank1;

StatementMeta(, 3314ae33-796b-4660-b656-cce564ee428f, 4, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [36]:
/* check -- do not use -- just a cross check */ 
select  count(distinct a.QuoteQueryGuid) 
from    stg.A1_MotorAcquisitionQuoteInitiated   a
JOIN 
        stg.A3_SL10_QQGUIDS                     z
            ON  a.QuoteQueryGuid = z.QuoteQueryGuid 


StatementMeta(, 3314ae33-796b-4660-b656-cce564ee428f, 37, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

StatementMeta(, 3314ae33-796b-4660-b656-cce564ee428f, 75, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [80]:
-- A3 query-- we need to copy the SL10 quote query guids as the scraperflag is quite complex than defined in the framework / chill's explanations in the past /* teams message sent to Nikita for check */
select  count(distinct a.QuoteQueryGuid) 
from    stg.A1_MotorAcquisitionQuoteInitiated  a
join 
        stg.MFQInsurersQuoted   b  
            ON  a.QuoteQueryGuid = b.QuoteQueryGuid 
JOIN 
        stg.A3_SL10_QQGUIDS     z
            ON  a.QuoteQueryGuid = z.QuoteQueryGuid 
left join         
        (select QuoteId, ABPAction 
         from   dlk.MFQ_MON_Session 
         Where  (IsKnownBot = TRUE 
            or 
                case when ABPAction = '' then null else ABPAction end is not null)  
            and `TimeStamp` between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' ) c  
            ON  a.QuoteQueryGuid = c.QuoteId 
where   b.InsurersQuoted = 'Y' 
and     z.Tot_All_First = 1 /* source filter to be applied */
and     z.InsurersQuoted = 'Y' /* source filter to be applied */
and     c.QuoteId is null 
and     RetrieveSessionToken is null 
and     Step1DateTime is not null 

StatementMeta(, 3314ae33-796b-4660-b656-cce564ee428f, 81, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourceQuoteReference,    
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
    a.QuoteQueryGuid,
    2,
    'Motor',
    date_trunc('MINUTE', Step1DateTime),
    cast(Step1DateTime as date),
    'A3',
    'Motor Acquisition - Customer gets a price',
    'Quote'
-- A3 query-- we need to copy the SL10 quote query guids as the scraperflag is quite complex than defined in the framework / chill's explanations in the past /* teams message sent to Nikita for check */
from    stg.MFQInsurersQuoted a
join 
        stg.A1_MotorAcquisitionQuoteInitiated b  
            ON  a.QuoteQueryGuid = b.QuoteQueryGuid 
JOIN 
        stg.A3_SL10_QQGUIDS     z
            ON  a.QuoteQueryGuid = z.QuoteQueryGuid 
left join         
        (select QuoteId, ABPAction from dlk.MFQ_MON_Session Where IsKnownBot = TRUE or case when ABPAction = '' then null else ABPAction end is not null  and `$ExtractDateTime` between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' ) c  
            ON  a.QuoteQueryGuid = c.QuoteId 
where   b.InsurersQuoted = 'Y' 
and     z.Tot_All_First = 1 /* source filter to be applied */
and     z.InsurersQuoted = 'Y' /* source filter to be applied */
and     c.QuoteId is null 
and     RetrieveSessionToken is null 
and     Step1DateTime is not null 

In [87]:
select  count(distinct a.QuoteQueryGuid) 
from    stg.A1_MotorAcquisitionQuoteInitiated a  
JOIN 
        stg.A3_SL10_QQGUIDS                   b   
            ON  a.QuoteQueryGuid = b.QuoteQueryGuid         
Where   b.Tot_All_First = 1 /* source filter to be applied */
and     b.InsurersQuoted = 'N' /* source filter to be applied */
and     FirstStep5DateTime is not NULL

StatementMeta(, 3314ae33-796b-4660-b656-cce564ee428f, 88, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourceQuoteReference,    
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
    QuoteQueryGuid,
    2,
    'Motor',
    date_trunc('MINUTE', FirstStep5DateTime),
    cast(FirstStep5DateTime as date),
    'A3.F1',
    'Motor Acquisition - No price returned, told to call us',
    'Quote'
from    stg.A1_MotorAcquisitionQuoteInitiated a  
JOIN 
        stg.A3_SL10_QQGUIDS                   b   
            ON  a.QuoteQueryGuid = b.QuoteQueryGuid         
Where   b.Tot_All_First = 1 /* source filter to be applied */
and     b.InsurersQuoted = 'N' /* source filter to be applied */
and     FirstStep5DateTime is not NULL
    /* 
        what about quotes starting inthe current month but completing next month ? 
            think we should get them too -- els we will have monthly lumps 
    */ 


In [1]:
/* A3.B1.1 -- Come back for this */
select	count(distinct a.QuoteQueryGuid) 
from	stg.a1_motoracquisitionquoteinitiated a, 
        stg.a1_motoracquisitionquoteinitiated b 
Where	a.QuoteQueryGuid = b.RetrieveSessionToken
;


StatementMeta(, 97bb0081-c19c-4403-a24c-2ac562469552, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourceQuoteReference,    
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
    a.QuoteQueryGuid,
    2,
    'Motor',
    date_trunc('MINUTE', a.LastStep5DateTime),
    cast(a.LastStep5DateTime as date),
    'A3.B1',
    'Motor Acquisition - Quote retrieve and re-quote cycles',
    'Quote'
from    stg.A1_MotorAcquisitionQuoteInitiated a  
JOIN 
        stg.A1_MotorAcquisitionQuoteInitiated b
            on a.QuoteQueryGuid = b.RetrieveSessionToken
/* this counts a first quote retrieved second and then a third time etc */ 

In [1]:
/* HFQ / MFQ to Policy links to be derived properly - we are picking up from DWH for now */
Create or Replace Table stg.GlobalPoliciesSold as 
select  PolicyCode, s.Channel, s.ReportingSaleType, s.PolicyTypeGroup, QuoteQueryGuid, HEQReference, ReportingSaleDate 
from    edw.tbl_fact_policy_sales    s 
WHERE   s.EffectiveDate = '2026-07-31'
  AND   s.ReportingSaleCategory = 'Cat A1: Active Sale'
  AND   s.OrgID = 1
  AND   s.SaleCount = 1
  ;

StatementMeta(, f7ef2338-5b0e-4c77-8619-c6fa9cfedfe8, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [104]:
/* Renewals and sales to be derived properly - we are picking up from DWH for now */
/* Renewals are always derived for the month of renewals -- and their sales over time across months */ 
Create or Replace Table stg.R0_MotorPoliciesEligibleForRenewals as 
select  PolicyCode, LyPolicyRenewDateAdj as RenewalDate, TyPolicyCode , TyPolicyRenewDateAdj
from    edw.tbl_fact_policy_renewals    a 
Where   a. LyPolicyTypeGroup = 'Home' 
and     a.LYPolicyRenewDateAdj between '2026-07-01' and '2026-07-31' 
and     PolicyOfferNum = 1 
Group by   PolicyCode, LyPolicyRenewDateAdj, TyPolicyCode  , TyPolicyRenewDateAdj

StatementMeta(, 3314ae33-796b-4660-b656-cce564ee428f, 105, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [45]:
insert INTO
ods.EpisodeEventStream
(        
    SourceQuoteReference,    
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
    a.QuoteQueryGuid,
    2,
    'Motor',
    date_trunc('MINUTE', PaymentDate),
    cast(PaymentDate as date),
    'A4a',
    'Motor Acquisition - Customer buys ONLINE',
    'Quote'
-- A3 query-- we need to copy the SL10 quote query guids as the scraperflag is quite complex than defined in the framework / chill's explanations in the past /* teams message sent to Nikita for check */
from    stg.GlobalPoliciesSold      a,
        (select QuoteQueryGuid, max(PaymentDate) PaymentDate
        from    stg.MFQ_Quote_Payments 
        where   PaymentStatus like '%|Success, code: 00%' 
        and     PaymentDate between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
        and     PaymentType in (0,5)
        Group by QuoteQueryGuid) b 
Where   a.QuoteQueryGuid = b.QuoteQueryGuid
;

StatementMeta(, 3314ae33-796b-4660-b656-cce564ee428f, 46, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [12]:
        select  PaymentMechanism, count(*)
        from    stg.MFQ_Quote_Payments 
        where   PaymentStatus like '%|Success, code: 00%' 
        and     PaymentDate between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
        Group by PaymentMEchanism
        limit 10 

StatementMeta(, 1556f380-0f59-41d3-8011-760d10cb6184, 13, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

In [97]:
        /* this is not used - MFQ payment to Web Assist resolution needs more analysis -- one for Vijay for Monday */
select  count(distinct PolicyCode), s.Channel 
from    stg.GlobalPoliciesSold      s
left join
        (select QuoteQueryGuid, max(PaymentDate) PaymentDate
        from    stg.MFQ_Quote_Payments 
        where   PaymentStatus like '%|Success, code: 00%' /* remove date filter to eliminate pre or post month purchases */ 
        and     PaymentType in (0,5)
        Group by QuoteQueryGuid) b 
        on   s.QuoteQueryGuid = b.QuoteQueryGuid
Where   b.QuoteQueryGuid is null 
and     s.ReportingSaleType = 'New Business' 
and     s.PolicyTypeGroup = 'Motor'
and     s.QuoteQueryGuid is not null 
Group by Channel
limit 100 

StatementMeta(, 3314ae33-796b-4660-b656-cce564ee428f, 98, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [ ]:
select  Channel, count(distinct PolicyCode), count(distinct QuoteQueryGuid)
from    stg.GlobalPoliciesSold      s
Where   s.ReportingSaleType = 'New Business' 
and     s.PolicyTypeGroup = 'Motor'
and     s.QuoteQueryGuid is not null 
Group by Channel
limit 100 

In [ ]:
select  Channel, count(distinct PolicyCode), count(distinct QuoteQueryGuid)
from    stg.GlobalPoliciesSold      s
Where   s.ReportingSaleType = 'New Business' 
and     s.PolicyTypeGroup = 'Motor'
and     s.QuoteQueryGuid is null 
Group by Channel
limit 100 

In [1]:
select  Channel, count(distinct PolicyCode), count(distinct a.QuoteQueryGuid) -- For these QuoteQueryGuids we need to count the chain back to see how many quotes they did in reality
from    stg.GlobalPoliciesSold  a, stg.A1_MotorAcquisitionQuoteInitiated b 
Where   a.QuoteQueryGuid = b.QuoteQueryGuid 
and     b.RetrieveSessionToken Is not Null 
and     a.ReportingSaleType = 'New Business' 
and     PolicyTypeGroup = 'Motor'
Group by Channel

StatementMeta(, b25fa0b7-064d-4fee-b528-de299c1c3e5f, 2, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 3 fields>

In [10]:
select  count(distinct PolicyCode), count(distinct a.QuoteQueryGuid) -- For these QuoteQueryGuids we need to count the chain back to see how many quotes they did in reality
from    stg.GlobalPoliciesSold  a, stg.A1_MotorAcquisitionQuoteInitiated b 
Where   a.QuoteQueryGuid = b.QuoteQueryGuid 
and     b.RetrieveSessionToken Is not Null 
and     PolicyTypeGroup = 'Motor'

StatementMeta(, 1556f380-0f59-41d3-8011-760d10cb6184, 11, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,    
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
    a.PolicyCode,
    9,
    'Motor',
    date_trunc('MINUTE', ReportingSaleDate),
    cast(ReportingSaleDate as date),
    'A4b',
    'Motor Acquisition - Agent quotes and converts on the CALL',
    'Policy'
from    stg.GlobalPoliciesSold      a
Where   a.ReportingSaleType = 'New Business' 
and     a.PolicyTypeGroup = 'Motor'
and     a.QuoteQueryGuid is null 
;


In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourceQuoteReference,
    SourcePolicyReference,    
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
    a.QuoteQueryGuid,
    a.PolicyCode,
    9,
    'Motor',
    date_trunc('MINUTE', ReportingSaleDate),
    cast(ReportingSaleDate as date),
    'A4.F1',
    'Motor Acquisition - Web assist: starts online, then calls to complete',
    'Policy' /* can be quote or policy! */
from    stg.GlobalPoliciesSold      a
Where   a.ReportingSaleType = 'New Business' 
and     a.PolicyTypeGroup = 'Motor'
and     a.QuoteQueryGuid is not null 
and     a.Channel = 'b) Web Assist'
;

In [107]:
select  count(distinct QuoteQueryGuid)
from    stg.MFQ_Quote_Payments 
where   PaymentDate between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 

StatementMeta(, 3314ae33-796b-4660-b656-cce564ee428f, 108, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourceQuoteReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
    a.QuoteQueryGuid,
    2,
    'Motor',
    date_trunc('MINUTE', PaymentDate),
    cast(PaymentDate as date),
    'A5a',
    'Motor Acquisition - Payment attempted ONLINE',
    'Quote'
from    stg.MFQ_Quote_Payments  a
where   PaymentDate between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
;

In [108]:
select  count(distinct QuoteQueryGuid)
from    stg.MFQ_Quote_Payments  a
where   PaymentDate between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     PaymentType not in (0,5)
;

StatementMeta(, 3314ae33-796b-4660-b656-cce564ee428f, 109, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourceQuoteReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
    a.QuoteQueryGuid,
    2,
    'Motor',
    date_trunc('MINUTE', PaymentDate),
    cast(PaymentDate as date),
    'A5.F1',
    'Motor Acquisition - Payment failed',
    'Quote'
from    stg.MFQ_Quote_Payments  a
where   PaymentDate between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     PaymentType not in (0,5)
;

In [19]:
/* A5.B1 Loan requested in MFQ payments */ 
select  count(distinct QuoteQueryGuid) 
from    stg.MFQ_Quote_Payments
Where   PaymentDate between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     PaymentType in (0,5)
and     FullPaymentFlag is FALSE 

StatementMeta(, 4958b5da-bf22-49e2-b5ac-26fc149e995a, 20, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourceQuoteReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
    QuoteQueryGuid,
    2,
    'Motor',
    date_trunc('MINUTE', PaymentDate),
    cast(PaymentDate as date),
    'A5.B1',
    'Motor Acquisition - Loan setup with the finance provider',
    'Quote'
from    stg.MFQ_Quote_Payments
Where   PaymentDate between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     PaymentType in (0,5)
and     FullPaymentFlag is FALSE 
;

In [22]:
select count(*) from dlk.SQL07CDA_Messages

StatementMeta(, 4958b5da-bf22-49e2-b5ac-26fc149e995a, 23, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [1]:
create or Replace table stg.CDAPurchasePolicyList as 
select  trim(PolicyNumber) PolicyCode , max(UTCDateUpdated) SentDate
from    dlk.SQL07CDA_Messages
where	RenderedMessage like 'Hi, thanks for choosing us, we really appreciate it.%be sending out your welcome pack shortly. Thanks, Chill Insurance.' 
and		UTCDateAdded	between  '2026-07-01 00:00:00' and '2026-08-01 00:00:00' 
and     ProviderResponse = 'Delivered'
Group by trim(PolicyNumber)

StatementMeta(, fdb1f955-81d8-4e98-a72d-cf144d5488e0, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [37]:
select  PolicyTypeGroup, count(*) 
from    stg.GlobalPoliciesSold a, stg.CDAPurchasePolicyList b 
Where   a.PolicyCode = b.PolicyCode
and     ReportingSaleType = 'New Business' 
Group by PolicyTypeGroup

StatementMeta(, 4958b5da-bf22-49e2-b5ac-26fc149e995a, 38, Finished, Available, Finished, False)

<Spark SQL result set with 3 rows and 2 fields>

In [ ]:
--EHN 20260901 - where is this coming from? There's the other A5.1 insert below, which is correct?

insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
        a.PolicyCode,
        1,
        'Motor',
        date_trunc('MINUTE', SentDate),
        cast(SentDate as date),
        'A5.1',
        'Motor Acquisition - Purchase confirmation sent',
        'Policy'
from    stg.GlobalPoliciesSold a, stg.CDAPurchasePolicyList b  /* the message is already filtered for new business */
Where   a.PolicyCode = b.PolicyCode
and     ReportingSaleType = 'New Business' 
and     PolicyTypeGroup = 'Motor'
;

In [2]:
/* Service Retrieve A2.E1 email 

    Ewan - please add this to event stream 
         
        
        so we may have to match to email / phone with pii and then to ods.scv and the scv key is to be pulled? 
*/
/*
select	*
from	dlk.EXT_XtremePushResults   a 
where	`timestamp` between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and		campaign_name = 'Motor - TYR 2.2 - Service Retrieve - Quote Expiry Journey - 0DTI' and interaction_type = 'sent' 
-- without the below conditions we get 76836 entries but it includes workfl/ sms / email so I think only 2/3 rd contacts sent out
and MessageType = 'EMAIL'  --> this brings down to 25269
and QuoteQueryGuid > ''  --> this brings down to 13339 these 2 conditions are a bit iffy as the volumes drop a lot
limit 100 

the full set of campaign name for July 
Motor - TYR 2.2 - Service Retrieve - Quote Expiry Journey - 0DTI
Motor - TYR 3.0 - Step 3 Complete
Motor - TYR 3.0 - Step 4 Complete
Motor - TYR 3.0 - Step 5 Complete
Motor TYR 2.2 - Buy Now - Service 
Motor TYR 2.2 - Buy Now - Service  AB TEST
Motor TYR 2.2 - Complete Quote - Marketing 
Motor TYR 2.2 - Complete Quote - Service 
Motor TYR 2.2 Buy Now - Marketing 

*/
select	count(*)
from	dlk.EXT_XtremePushResults   a 
where	`timestamp` between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and		campaign_name like 'Motor % TYR %' 
and interaction_type = 'sent' 
-- without the below conditions we get 70113 entries but it includes workflow/ sms / email so I think only 2/3 rd contacts sent out
and MessageType in ('SMS', 'EMAIL')  --> this brings down to 23058
-- and QuoteQueryGuid > ''  --> this brings down to 12148 these 2 conditions are a bit iffy as the volumes drop a lot
limit 100 

StatementMeta(, 1556f380-0f59-41d3-8011-760d10cb6184, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [7]:
select 
--QuoteQueryGuid,count(*) 
--count(*)
*
from dlk.EXT_XtremePushResults 
where	`timestamp` between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and		campaign_name like 'Motor % TYR %' 
and     interaction_type = 'sent' 
and     MessageType = 'EMAIL'
and     QuoteQueryGuid > ''
--and     QuoteQueryGuid = 'a504caec-6a11-492b-a6e6-de2f874e9467'
--group by QuoteQueryGuid
--having count(*) > 1 
limit 10

StatementMeta(, 513eeba3-c789-4544-813e-13631d9ec9b5, 8, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 17 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
        a.QuoteQueryGuid,
        1,
        'Motor',
        date_trunc('MINUTE', `timestamp`),
        cast(`timestamp` as date),
        'A1.E1',
        'Motor Acquisition - Service retrieve',
        'Policy'
from    dlk.EXT_XtremePushResults
where	`timestamp` between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and		campaign_name like 'Motor % TYR %' 
and     interaction_type = 'sent' 
and     MessageType = 'EMAIL'
and     QuoteQueryGuid > ''
;

In [4]:
/* purchase confirmation email from Amber A5.1 - email confirmation -- this pulls 2890 ecs and aligns with Nikita's number


    Ewan - please add this to event stream 
        
        but policy code / client code is sparsely populated 
        
        so we may have to match to email / phone with pii and then to ods.scv and the scv key is to be pulled? */

select	count(distinct PolicyCode), count(distinct user_id)
--*
from	dlk.EXT_XtremePushResults
where	`timestamp` between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and		campaign_name = 'Online Motor Bind Confirmation' and interaction_type = 'sent' 
--and     QuoteQueryGuid > ''
limit 10

StatementMeta(, 1556f380-0f59-41d3-8011-760d10cb6184, 5, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [ ]:
--TBC

insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
        a.QuoteQueryGuid,
        1,
        'Motor',
        date_trunc('MINUTE', `timestamp`),
        cast(`timestamp` as date),
        'A5.1',
        'Motor Acquisition - Service retrieve',
        'Policy'
from    dlk.EXT_XtremePushResults
where	`timestamp` between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and		campaign_name = 'Online Motor Bind Confirmation' and interaction_type = 'sent' 
;

In [7]:
/* docs asked for  */
select  
		count(distinct PolicyCode) 
from edw.EXP_MyChill_Chase_Daily_Snapshot_MotorVan a
Where	coalesce(try_cast(concat(right(SaleDate,4), '-',substring(SaleDate,4,2),'-',left(SaleDate,2)) as date),'1900-01-01') between '2026-07-01' and '2026-07-31' 
and PolicyType = 'New Business' 
and PolicyTypeGroup = 'Motor' 
and case 
        when Sign_Prop_Status				= 'O' then 1 
        when Sign_Stat_Of_Fact_Status		= 'O' then 1 
        when Comp_Fin_Agree_Status			= 'O' then 1 
        when Comp_DDM_Status				= 'O' then 1 
        when Proof_Of_NCB_Status			= 'O' then 1 
        when Trans_Proof_Of_NCB_Status		= 'O' then 1 
        when Ltr_Of_Driv_Exp_Status			= 'O' then 1 
        when Prop_Driv_Lic_Status			= 'O' then 1 
        when Nam_Driv_Lic_Status			= 'O' then 1 
        when Gap_In_Cov_Ltr_Status			= 'O' then 1 
        when `2ndCar_Cert_Status`			= 'O' then 1 
        when Doc_Ltr_Status					= 'O' then 1 
        when Engineers_Rpt_Status			= 'O' then 1 
        when NCT_Status						= 'O' then 1 
        when Veh_Lic_Cert_Status			= 'O' then 1 
        when Irish_Reg_Status				= 'O' then 1 
        when Main_Driver_Dec_Status			= 'O' then 1 
        when Soc_Dom_Pleas_Dec_Status		= 'O' then 1 
        when Comp_Car_Exper_Status			= 'O' then 1 
        when DD_Conf_Ltr_Status				= 'O' then 1 
        when Comp_Fin_Mand_Status			= 'O' then 1 
        when Val_For_Spec_Item_Status		= 'O' then 1 
        when Dri_Lic_Num_Status				= 'O' then 1 
        when Comp_Lost_Cert_Dec_Status		= 'O' then 1 
        when Orig_Cert_Status				= 'O' then 1 
        when Cancel_Req_Status				= 'O' then 1 
        when PPS_Num_Status					= 'O' then 1 
        when Afford_State_Status			= 'O' then 1 
        when Identification_Status			= 'O' then 1 
        when Digital_Journey_Status			= 'O' then 1 
        when Finance_Form_Status			= 'O' then 1 
        else 0
    End = 1 
and Campaign = 'DAY 1'

StatementMeta(, 1556f380-0f59-41d3-8011-760d10cb6184, 8, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [6]:
/* not supplied ever */
select  
		count(distinct PolicyCode) 
from edw.EXP_MyChill_Chase_Daily_Snapshot_MotorVan a
Where	coalesce(try_cast(concat(right(SaleDate,4), '-',substring(SaleDate,4,2),'-',left(SaleDate,2)) as date),'1900-01-01') between '2026-07-01' and '2026-07-31' 
and PolicyType = 'New Business' 
and PolicyTypeGroup = 'Motor' 
and case 
        when Sign_Prop_Status				= 'O' then 1 
        when Sign_Stat_Of_Fact_Status		= 'O' then 1 
        when Comp_Fin_Agree_Status			= 'O' then 1 
        when Comp_DDM_Status				= 'O' then 1 
        when Proof_Of_NCB_Status			= 'O' then 1 
        when Trans_Proof_Of_NCB_Status		= 'O' then 1 
        when Ltr_Of_Driv_Exp_Status			= 'O' then 1 
        when Prop_Driv_Lic_Status			= 'O' then 1 
        when Nam_Driv_Lic_Status			= 'O' then 1 
        when Gap_In_Cov_Ltr_Status			= 'O' then 1 
        when `2ndCar_Cert_Status`			= 'O' then 1 
        when Doc_Ltr_Status					= 'O' then 1 
        when Engineers_Rpt_Status			= 'O' then 1 
        when NCT_Status						= 'O' then 1 
        when Veh_Lic_Cert_Status			= 'O' then 1 
        when Irish_Reg_Status				= 'O' then 1 
        when Main_Driver_Dec_Status			= 'O' then 1 
        when Soc_Dom_Pleas_Dec_Status		= 'O' then 1 
        when Comp_Car_Exper_Status			= 'O' then 1 
        when DD_Conf_Ltr_Status				= 'O' then 1 
        when Comp_Fin_Mand_Status			= 'O' then 1 
        when Val_For_Spec_Item_Status		= 'O' then 1 
        when Dri_Lic_Num_Status				= 'O' then 1 
        when Comp_Lost_Cert_Dec_Status		= 'O' then 1 
        when Orig_Cert_Status				= 'O' then 1 
        when Cancel_Req_Status				= 'O' then 1 
        when PPS_Num_Status					= 'O' then 1 
        when Afford_State_Status			= 'O' then 1 
        when Identification_Status			= 'O' then 1 
        when Digital_Journey_Status			= 'O' then 1 
        when Finance_Form_Status			= 'O' then 1 
        else 0
    End = 1 
and Campaign = 'DAY 20'

StatementMeta(, 1556f380-0f59-41d3-8011-760d10cb6184, 7, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [2]:
select * from edw.EXP_MyChill_Chase_Daily_Snapshot_MotorVan 
where MobileNo = '353868356106'
--limit 10

SELECT <non-pivoted columns>, [<pivoted column 1>], [<pivoted column 2>],
FROM (
SELECT PolicyCode
FROM edw.EXP_MyChill_Chase_Daily_Snapshot_MotorVan 
) AS SourceTable
PIVOT (
<aggregation_function>(<column_to_aggregate>)
FOR <column_to_pivot> IN ([<pivoted column 1>], [<pivoted column 2>], ...)
) AS PivotTable;

StatementMeta(, 4a4eaa61-77cc-4ce1-9013-77dcc43daa19, 3, Finished, Available, Finished, False)

<Spark SQL result set with 4 rows and 87 fields>

In [14]:
/* A6.B1.1 is not recorded anywhere - the document checklist sms / email is all we can see -- A6 */
/* A6.B1.2 is here */
select  --count(distinct b.PolicyCode)
*
from    stg.GlobalPoliciesSold a, dlk.MyChill_NewUploadDocumentEvents  b 
Where   a.PolicyCode = b.PolicyCode
and     ReportingSaleType = 'New Business' 
and     PolicyTypeGroup = 'Motor'
and     `Timestamp` > '2026-07-01 00:00:00.000' 
limit 100
;

StatementMeta(, 513eeba3-c789-4544-813e-13631d9ec9b5, 15, Finished, Available, Finished, False)

<Spark SQL result set with 100 rows and 20 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
        a.PolicyCode,
        1,
        'Motor',
        date_trunc('MINUTE', `timestamp`),
        cast(`timestamp` as date),
        'A6.B1.2',
        'Motor Acquisition - Customer submits documents',
        'Policy'
from    stg.GlobalPoliciesSold a, dlk.MyChill_NewUploadDocumentEvents  b 
Where   a.PolicyCode = b.PolicyCode
and     ReportingSaleType = 'New Business' 
and     PolicyTypeGroup = 'Motor'
and     `Timestamp` > '2026-07-01 00:00:00.000' 
;

In [12]:
/* A6.B1.3 */
select  isAccepted, count(distinct b.PolicyCode)
from    stg.GlobalPoliciesSold a, dlk.MyChillWorkflow_DocumentStatus  b 
Where   a.PolicyCode = b.PolicyCode
and     ReportingSaleType = 'New Business' 
and     PolicyTypeGroup = 'Motor'
and     `Timestamp` > '2026-07-01 00:00:00.000' 
/*and     isAccepted IsRejected */
Group by isAccepted
limit 100
;

StatementMeta(, 139d79e9-1b83-48a6-bd9c-264791ed9e36, 13, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
        a.PolicyCode,
        1,
        'Motor',
        date_trunc('MINUTE', `timestamp`),
        cast(`timestamp` as date),
        'A6.B1.3',
        'Motor Acquisition - Customer submits documents',
        'Policy'
from    stg.GlobalPoliciesSold a, dlk.MyChillWorkflow_DocumentStatus  b 
Where   a.PolicyCode = b.PolicyCode
and     ReportingSaleType = 'New Business' 
and     PolicyTypeGroup = 'Motor'
and     `Timestamp` > '2026-07-01 00:00:00.000' 
and     isAccepted = 'true'
;

In [2]:
select  *from dlk.cda_loanse

StatementMeta(, d8dc7ba4-37bb-440b-870d-46271665ae32, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 23 fields>

In [3]:
select count(*) from dlk.rbsdata2_diary

StatementMeta(, d8dc7ba4-37bb-440b-870d-46271665ae32, 4, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [1]:
/* A6.B1.1 A6 */
SELECT  count(distinct a.PolicyCode)
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in 
            (
                'New Business - Saved Document - Document Checklist SMS',
                'New Business - Saved Document - Document Checklist Email',
                'New Business - Saved Document - Terms Of Business',    
                'New Business - Sent To Document Processing - Terms Of Business'
            )


StatementMeta(, d5042f13-38fc-46ce-b16d-50aeacbe2afe, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
        a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', EventDateTime)),
        max(cast(EventDate as date)),
        'A6',
        'Motor Acquisition - Documents requested',
        'Policy'
FROM    edw.EXP_MyChill_Chase_Daily_Snapshot_MotorVan a
Where	coalesce(try_cast(concat(right(SaleDate,4), '-',substring(SaleDate,4,2),'-',left(SaleDate,2)) as date),'1900-01-01') between '2026-07-01' and '2026-07-31' and PolicyType = 'New Business' and PolicyTypeGroup = 'Motor' and 
case 
	when Sign_Prop_Status				= 'O' then 1 
	when Sign_Stat_Of_Fact_Status		= 'O' then 1 
	when Comp_Fin_Agree_Status			= 'O' then 1 
	when Comp_DDM_Status				= 'O' then 1 
	when Proof_Of_NCB_Status			= 'O' then 1 
	when Trans_Proof_Of_NCB_Status		= 'O' then 1 
	when Ltr_Of_Driv_Exp_Status			= 'O' then 1 
	when Prop_Driv_Lic_Status			= 'O' then 1 
	when Nam_Driv_Lic_Status			= 'O' then 1 
	when Gap_In_Cov_Ltr_Status			= 'O' then 1 
	when [2ndCar_Cert_Status]			= 'O' then 1 
	when Doc_Ltr_Status					= 'O' then 1 
	when Engineers_Rpt_Status			= 'O' then 1 
	when NCT_Status						= 'O' then 1 
	when Veh_Lic_Cert_Status			= 'O' then 1 
	when Irish_Reg_Status				= 'O' then 1 
	when Main_Driver_Dec_Status			= 'O' then 1 
	when Soc_Dom_Pleas_Dec_Status		= 'O' then 1 
	when Comp_Car_Exper_Status			= 'O' then 1 
	when DD_Conf_Ltr_Status				= 'O' then 1 
	when Comp_Fin_Mand_Status			= 'O' then 1 
	when Val_For_Spec_Item_Status		= 'O' then 1 
	when Dri_Lic_Num_Status				= 'O' then 1 
	when Comp_Lost_Cert_Dec_Status		= 'O' then 1 
	when Orig_Cert_Status				= 'O' then 1 
	when Cancel_Req_Status				= 'O' then 1 
	when PPS_Num_Status					= 'O' then 1 
	when Afford_State_Status			= 'O' then 1 
	when Identification_Status			= 'O' then 1 
	when Digital_Journey_Status			= 'O' then 1 
	when Finance_Form_Status			= 'O' then 1 
	else 0
End = 1 
and Campaign = 'DAY 1'

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
        a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', EventDateTime)),
        max(cast(EventDate as date)),
        'A6.F1',
        'Motor Acquisition - Documents requested',
        'Policy'
FROM    edw.EXP_MyChill_Chase_Daily_Snapshot_MotorVan a
Where	coalesce(try_cast(concat(right(SaleDate,4), '-',substring(SaleDate,4,2),'-',left(SaleDate,2)) as date),'1900-01-01') between '2026-07-01' and '2026-07-31' and PolicyType = 'New Business' and PolicyTypeGroup = 'Motor' and 
case 
	when Sign_Prop_Status				= 'O' then 1 
	when Sign_Stat_Of_Fact_Status		= 'O' then 1 
	when Comp_Fin_Agree_Status			= 'O' then 1 
	when Comp_DDM_Status				= 'O' then 1 
	when Proof_Of_NCB_Status			= 'O' then 1 
	when Trans_Proof_Of_NCB_Status		= 'O' then 1 
	when Ltr_Of_Driv_Exp_Status			= 'O' then 1 
	when Prop_Driv_Lic_Status			= 'O' then 1 
	when Nam_Driv_Lic_Status			= 'O' then 1 
	when Gap_In_Cov_Ltr_Status			= 'O' then 1 
	when `2ndCar_Cert_Status`			= 'O' then 1 
	when Doc_Ltr_Status					= 'O' then 1 
	when Engineers_Rpt_Status			= 'O' then 1 
	when NCT_Status						= 'O' then 1 
	when Veh_Lic_Cert_Status			= 'O' then 1 
	when Irish_Reg_Status				= 'O' then 1 
	when Main_Driver_Dec_Status			= 'O' then 1 
	when Soc_Dom_Pleas_Dec_Status		= 'O' then 1 
	when Comp_Car_Exper_Status			= 'O' then 1 
	when DD_Conf_Ltr_Status				= 'O' then 1 
	when Comp_Fin_Mand_Status			= 'O' then 1 
	when Val_For_Spec_Item_Status		= 'O' then 1 
	when Dri_Lic_Num_Status				= 'O' then 1 
	when Comp_Lost_Cert_Dec_Status		= 'O' then 1 
	when Orig_Cert_Status				= 'O' then 1 
	when Cancel_Req_Status				= 'O' then 1 
	when PPS_Num_Status					= 'O' then 1 
	when Afford_State_Status			= 'O' then 1 
	when Identification_Status			= 'O' then 1 
	when Digital_Journey_Status			= 'O' then 1 
	when Finance_Form_Status			= 'O' then 1 
	else 0
End = 1 
and Campaign = 'DAY 20'

In [14]:
/* A6.B1.5 */
SELECT  count(distinct a.PolicyCode)
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription like '%Chase%' 
and     EventDescription like '%Final%' 

StatementMeta(, 139d79e9-1b83-48a6-bd9c-264791ed9e36, 15, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
        a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', EventDateTime)),
        max(cast(EventDate as date)),
        'A6.B1.5',
        'Motor Acquisition - Documents requested',
        'Policy'
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription like '%Chase%' 
and     EventDescription like '%Final%' 

In [19]:
/* A6.B1.O1 -- no doc escalation and then cancellation */
SELECT  count(distinct a.PolicyCode)
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in 
        (
            'Insurer Led Cancelation - Emailed Document - Reg canx email template',
            'Insurer Led Cancelation - Sent To Document Processing - Reg canx email template'
        )
and     a.PolicyCode in 
        (SELECT  distinct a.PolicyCode
        FROM    stg.GlobalPoliciesSold                   a,
                ods.EventStream                          d 
        Where   a.PolicyCode = d.SourcePolicyReference
        and     d.EventSourceId = 3 
        and     d.PolicyTypeGroup = 'Motor'
        and     EventDateTime > '2026-07-01 00:00:00.000' 
        and     ReportingSaleType = 'New Business' 
        and     EventDescription like '%Chase%' 
        and     EventDescription like '%Final%' ) 

StatementMeta(, 139d79e9-1b83-48a6-bd9c-264791ed9e36, 20, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT
        a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', EventDateTime)),
        max(cast(EventDate as date)),
        'A6.B1.O1',
        'Motor Acquisition - Documents requested',
        'Policy'
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in 
        (
            'Insurer Led Cancelation - Emailed Document - Reg canx email template',
            'Insurer Led Cancelation - Sent To Document Processing - Reg canx email template'
        )
and     a.PolicyCode in 
        (SELECT  distinct a.PolicyCode
        FROM    stg.GlobalPoliciesSold                   a,
                ods.EventStream                          d 
        Where   a.PolicyCode = d.SourcePolicyReference
        and     d.EventSourceId = 3 
        and     d.PolicyTypeGroup = 'Motor'
        and     EventDateTime > '2026-07-01 00:00:00.000' 
        and     ReportingSaleType = 'New Business' 
        and     EventDescription like '%Chase%' 
        and     EventDescription like '%Final%' ) 

In [24]:
/* A7 */
SELECT  count(distinct a.PolicyCode)
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in 
            ( 
                'New Business - Document Transmission Confirmed - Certificate',
                'New Business - Document Posted Confirmation - Certificate',
                'New Business - Document Transmitted - Certificate'
            )   


StatementMeta(, 139d79e9-1b83-48a6-bd9c-264791ed9e36, 25, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', EventDateTime)),
        max(cast(EventDate as date)),
        'A7',
        'Motor Acquisition - Documents requested',
        'Policy'
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in 
            ( 
                'New Business - Document Transmission Confirmed - Certificate',
                'New Business - Document Posted Confirmation - Certificate',
                'New Business - Document Transmitted - Certificate'
            )   

In [30]:
/* A7 , A7.B1.1 */
SELECT  count(distinct a.PolicyCode)
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription IN 
            ( 
                'New Business - Document Transmitted - Certificate' /* as confirmed by 2003 / 2004 / 2014 only this exists --Document Transmitted */
            )   


StatementMeta(, 139d79e9-1b83-48a6-bd9c-264791ed9e36, 31, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', EventDateTime)),
        max(cast(EventDate as date)),
        'A7.B1.1',
        'Motor Acquisition - Documents requested',
        'Policy'
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription IN 
            ( 
                'New Business - Document Transmitted - Certificate' /* as confirmed by 2003 / 2004 / 2014 only this exists --Document Transmitted */
            )   

In [29]:
/* A7 , A7.B1.2 */
SELECT  count(distinct a.PolicyCode)
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in (
            'New Business - Document Transmitted - Terms Of Business',
            'New Business - Emailed Document - Terms Of Business'
            )


StatementMeta(, 139d79e9-1b83-48a6-bd9c-264791ed9e36, 30, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', EventDateTime)),
        max(cast(EventDate as date)),
        'A7.B1.2',
        'Motor Acquisition - Documents requested',
        'Policy'
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in (
            'New Business - Document Transmitted - Terms Of Business',
            'New Business - Emailed Document - Terms Of Business'
            )

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', EventDateTime)),
        max(cast(EventDate as date)),
        'A7.B1.3',
        'Motor Acquisition - Documents requested',
        'Policy'
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in (
            'New Business - Document Transmitted - Terms Of Business',
            'New Business - Emailed Document - Terms Of Business'
            )

In [31]:
/* A7 , A7.B1.2 */
SELECT  count(distinct a.PolicyCode)
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in (
            'Duplicate Certificate - Document Transmitted - Certificate'
            )

StatementMeta(, 139d79e9-1b83-48a6-bd9c-264791ed9e36, 32, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
insert INTO
ods.EpisodeEventStream
(        
    SourcePolicyReference,
    SourceSystemId,          
    PolicyTypeGroup,         
    EventDateTime,           
    EventDate,               
    EventTypeId,     
    EventDescription,
    Grain
)
SELECT  a.PolicyCode,
        1,
        'Motor',
        max(date_trunc('MINUTE', EventDateTime)),
        max(cast(EventDate as date)),
        'A7.B1.5',
        'Motor Acquisition - Documents requested',
        'Policy'
FROM    stg.GlobalPoliciesSold                   a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Motor'
and     EventDateTime > '2026-07-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in (
            'Duplicate Certificate - Document Transmitted - Certificate'
            )

In [ ]:
-- get chase emails from XP table
select campaign_name from ods.EXT_XtremePushResults where campaign_name like '%Chase%' 
and [timestamp] between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
Group by campaign_name
/* EXT_XtremePushResults_Policy is at policycode level -- the above is email level */ 
/*
Arrears - Chase 1 - 1 Day Past
Arrears - Chase 2 - 7 Days ago
Arrears - Chase 3 - 14 Days Past
Arrears - Chase 4 - 21 Days Past
Arrears - Chase 5 - 28 Days Past
Home - Post Renewal &amp; Purchase - xSell 
Home - Renewals Chase - 10 DTI
Home - Renewals Chase - 10 DTI - Ex-Chrome
Home - Renewals Chase - 20DTI
Home - Renewals Chase - 20DTI - Ex-Chrome
Home - Renewals Chase - 2DTI
Home - Renewals Chase - 2DTI - Exh-Chrome
Motor - Post Renewal & Purchase - xSell
Motor - Renewals Chase - 10DTI
Motor - Renewals Chase - 20DTI
Motor - Renewals Chase - 2DTI
Renewal Chase 16 Fri
Renewal Chase 16 Mon-Thur
Renewal Chase 21 Fri
Renewal Chase 21 Mon-Thur
SurveyPurchased Auto PIF
SurveyPurchased July23
Van - Renewals Chase - 10DTI
Van - Renewals Chase - 20DTI
Van - Renewals Chase - 2DTI
*/

In [ ]:
select * from stg.XPTempExtracts where 'Commercial'

        create or replace table tmp.lifelivepolicies as
        select  cast(a.PK_ContactID as varchar(255)) ClientCode,
                b.PK_ContactPolicyID
        from    dlk.life_tblcontacts        a,
                dlk.life_tblcontactpolicies b
        Where   a.PK_ContactID = b.FK_ContactID
        and     FK_PolicyStatusID = 1
        and     PolicyInceptionDate < '2026-08-29 00:00:00.000'
        AND     PolicyInceptionDate <> '1900-01-01 00:00:00.000'
        AND     PolicyMaturityDate >= '2026-08-28'
        AND     PolicyNumber IS NOT NULL
        AND     PolicyNumber <> ''
        AND     PolicyNumber <> 'GDPR'
        Order by 1

In [1]:
select	* 
from	[msg].[Messages] 
where	RenderedMessage not like 'Hi, thanks for choosing us, we really appreciate it. We''ll be sending out your welcome pack shortly. Thanks, Chill Insurance.%' 
and		RenderedMessage not like 'Urgent! Your renewal is due tomorrow%'
and		RenderedMessage not like 'Urgent! Your renewal is due. %'
and		RenderedMessage not like 'Urgent! Your renewal is now due. %'
and		RenderedMessage <> 'Chill MFQ Chase'
and		RenderedMessage not like '%thank you for choosing Chill Insurance. We have issued your Cert and Disc.%'
and		RenderedMessage not like '%we have emailed important correspondence regarding your insurance policy %'
and		UTCDateAdded	between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
Limit 100 

StatementMeta(, 7f34cdd0-8f5e-454d-868f-ab2f23debe7f, 2, Finished, Available, Finished, False)

Error: 
[PARSE_SYNTAX_ERROR] Syntax error at or near '100'.(line 1, pos 11)

== SQL ==
select	top 100 * 
-----------^^^
from	[msg].[Messages] 
where	RenderedMessage not like 'Hi, thanks for choosing us, we really appreciate it. We''ll be sending out your welcome pack shortly. Thanks, Chill Insurance.%' 
and		RenderedMessage not like 'Urgent! Your renewal is due tomorrow%'
and		RenderedMessage not like 'Urgent! Your renewal is due. %'
and		RenderedMessage not like 'Urgent! Your renewal is now due. %'
and		RenderedMessage <> 'Chill MFQ Chase'
and		RenderedMessage not like '%thank you for choosing Chill Insurance. We have issued your Cert and Disc.%'
and		RenderedMessage not like '%we have emailed important correspondence regarding your insurance policy %'
and		UTCDateAdded	between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 

Hi Lorraine, we have emailed important correspondence regarding your insurance policy to lorrainemurphy76@hotmail.com. Thanks, Chill Insurance.
Hi Ann 